# Tarea 1: Modelado de Datos Privados en RDF

**Curso:** Web Semántica y Datos Abiertos  
**Universidad:** Universidad Técnica Federico Santa María  
**Profesor:** Jose Emilio Labra Gayo  
**Autores:** Kevin Cortes, Cristian Orellana, Ricardo Roman

---

## Objetivo
Modelar un conjunto de datos privados (ficticios) de la **Clínica Bienestar** en formato RDF/Turtle, utilizando vocabularios estándar como FOAF, Schema.org y un namespace propio.

## Descripción del Dataset
El dataset simula el **Registro de Pacientes** de la Clínica Bienestar, con información sobre:
- Pacientes (ID, nombre, fecha de nacimiento)
- Citas médicas (fecha, diagnóstico CIE-10)

## 1. Instalación de dependencias

In [35]:
# Instalar la librería rdflib para manipulación de grafos RDF
!pip install rdflib -q
print('✅ rdflib instalado correctamente')

✅ rdflib instalado correctamente


## 2. Clonar el repositorio y cargar los datos

In [36]:
import os

REPO_URL = 'https://github.com/corellanag/web-semantica-tarea-final.git'

# Si necesitas ejecutar comandos de shell, hazlo por separado:
import subprocess
import os

# Limpiar directorio existente si es necesario
if os.path.exists('web-semantica-tarea-final'):
    subprocess.run(['rm', '-rf', 'web-semantica-tarea-final'])

# Clonar el repositorio
subprocess.run(['git', 'clone', REPO_URL])

CompletedProcess(args=['git', 'clone', 'https://github.com/corellanag/web-semantica-tarea-final.git'], returncode=0)

## 3. Definición de Vocabularios y Namespaces

In [37]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, FOAF

# Definición de namespaces
SDO    = Namespace('https://schema.org/')
CLINIC = Namespace('http://www.clinica-bienestar.es/ontology#')
PAT    = Namespace('http://www.clinica-bienestar.es/data/patient/')
APPT   = Namespace('http://www.clinica-bienestar.es/data/appointment/')

print('Namespaces definidos:')
print(f'  SDO    = {SDO}')
print(f'  CLINIC = {CLINIC}')
print(f'  PAT    = {PAT}')
print(f'  APPT   = {APPT}')

Namespaces definidos:
  SDO    = https://schema.org/
  CLINIC = http://www.clinica-bienestar.es/ontology#
  PAT    = http://www.clinica-bienestar.es/data/patient/
  APPT   = http://www.clinica-bienestar.es/data/appointment/


## 4. Construcción del Grafo RDF desde cero (programático)

In [38]:
from datetime import date, datetime

# Crear el grafo
g = Graph()

# Vincular prefijos para serialización legible
g.bind('rdf',         RDF)
g.bind('rdfs',        RDFS)
g.bind('xsd',         XSD)
g.bind('foaf',        FOAF)
g.bind('sdo',         SDO)
g.bind('clinic',      CLINIC)
g.bind('patient',     PAT)
g.bind('appointment', APPT)

# -------------------------------------------------------
# Datos de pacientes y citas
# -------------------------------------------------------
pacientes = [
    {'uri': 'P7821A', 'id': 'P7821A', 'nombre': 'Carlos Sánchez',  'nacimiento': '1978-04-12',
     'citas': [('c20250915', '2025-09-15T10:00:00', 'I10'),
               ('c20260120', '2026-01-20T11:30:00', 'J06.9')]},
    {'uri': 'P9345B', 'id': 'P9345B', 'nombre': 'Laura Martínez',  'nacimiento': '1990-07-23',
     'citas': [('c20260122', '2026-01-22T09:15:00', 'E11')]},
    {'uri': 'P4412C', 'id': 'P4412C', 'nombre': 'Miguel Torres',   'nacimiento': '1965-11-05',
     'citas': [('c20260205', '2026-02-05T08:30:00', 'M54.5'),
               ('c20260301', '2026-03-01T10:00:00', 'M54.5')]},
    {'uri': 'P6678D', 'id': 'P6678D', 'nombre': 'Ana García',      'nacimiento': '1983-02-18',
     'citas': [('c20260210', '2026-02-10T14:00:00', 'F32.0')]},
    {'uri': 'P3301E', 'id': 'P3301E', 'nombre': 'Pedro Ramírez',   'nacimiento': '1955-09-30',
     'citas': [('c20260215', '2026-02-15T09:00:00', 'I25.1'),
               ('c20260228', '2026-02-28T11:00:00', 'I10')]},
]

# Agregar triples al grafo
for p in pacientes:
    pat_uri = PAT[p['uri'].lower()]
    g.add((pat_uri, RDF.type,           CLINIC.Paciente))
    g.add((pat_uri, CLINIC.idPaciente,  Literal(p['id'],       datatype=XSD.string)))
    g.add((pat_uri, FOAF.name,          Literal(p['nombre'],   datatype=XSD.string)))
    g.add((pat_uri, SDO.birthDate,      Literal(p['nacimiento'], datatype=XSD.date)))

    for cita_id, fecha, diag in p['citas']:
        cita_uri = APPT[cita_id]
        g.add((pat_uri,   CLINIC.tieneCita,      cita_uri))
        g.add((cita_uri,  RDF.type,              CLINIC.Cita))
        g.add((cita_uri,  CLINIC.fechaCita,      Literal(fecha + 'Z', datatype=XSD.dateTime)))
        g.add((cita_uri,  CLINIC.conDiagnostico, Literal(diag,        datatype=XSD.string)))

print(f'✅ Grafo construido con {len(g)} triples')

✅ Grafo construido con 52 triples


## 5. Visualizar el grafo en formato Turtle

In [39]:
turtle_output = g.serialize(format='turtle')
print(turtle_output)

@prefix appointment: <http://www.clinica-bienestar.es/data/appointment/> .
@prefix clinic: <http://www.clinica-bienestar.es/ontology#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix patient: <http://www.clinica-bienestar.es/data/patient/> .
@prefix sdo: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

patient:p3301e a clinic:Paciente ;
    clinic:idPaciente "P3301E"^^xsd:string ;
    clinic:tieneCita appointment:c20260215,
        appointment:c20260228 ;
    foaf:name "Pedro Ramírez"^^xsd:string ;
    sdo:birthDate "1955-09-30"^^xsd:date .

patient:p4412c a clinic:Paciente ;
    clinic:idPaciente "P4412C"^^xsd:string ;
    clinic:tieneCita appointment:c20260205,
        appointment:c20260301 ;
    foaf:name "Miguel Torres"^^xsd:string ;
    sdo:birthDate "1965-11-05"^^xsd:date .

patient:p6678d a clinic:Paciente ;
    clinic:idPaciente "P6678D"^^xsd:string ;
    clinic:tieneCita appointment:c20260210 ;
    foaf:name "Ana García"^^xsd:string ;
    sdo:

## 6. Cargar el archivo TTL del repositorio y comparar

In [40]:
# Cargar el archivo Turtle del repositorio
g_file = Graph()
g_file.parse('web-semantica-tarea-final/private/clinica_bienestar.ttl', format='turtle')
print(f'✅ Grafo cargado desde archivo: {len(g_file)} triples')

# Mostrar todos los sujetos únicos
print('\nRecursos en el grafo:')
for s in set(g_file.subjects()):
    print(f'  {s}')

✅ Grafo cargado desde archivo: 74 triples

Recursos en el grafo:
  http://www.clinica-bienestar.es/data/appointment/c20260301
  http://www.clinica-bienestar.es/ontology#Cita
  http://www.clinica-bienestar.es/data/patient/3301e
  http://www.clinica-bienestar.es/data/appointment/c20250915
  http://www.clinica-bienestar.es/data/patient/7821a
  http://www.clinica-bienestar.es/data/patient/9345b
  http://www.clinica-bienestar.es/ontology#fechaCita
  http://www.clinica-bienestar.es/data/patient/6678d
  http://www.clinica-bienestar.es/data/appointment/c20260205
  http://www.clinica-bienestar.es/ontology#tieneCita
  http://www.clinica-bienestar.es/ontology#conDiagnostico
  http://www.clinica-bienestar.es/data/appointment/c20260122
  http://www.clinica-bienestar.es/data/patient/4412c
  http://www.clinica-bienestar.es/ontology#Paciente
  http://www.clinica-bienestar.es/data/appointment/c20260215
  http://www.clinica-bienestar.es/data/appointment/c20260120
  http://www.clinica-bienestar.es/data/a

## 7. Consultas SPARQL básicas sobre el grafo

In [41]:
# Consulta 1: Listar todos los pacientes
query_pacientes = """
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX clinic: <http://www.clinica-bienestar.es/ontology#>

SELECT ?id ?nombre
WHERE {
  ?p a clinic:Paciente ;
     clinic:idPaciente ?id ;
     foaf:name ?nombre .
}
ORDER BY ?id
"""

print('=== Pacientes registrados ===')
for row in g_file.query(query_pacientes):
    print(f'  ID: {row.id} | Nombre: {row.nombre}')

=== Pacientes registrados ===
  ID: P3301E | Nombre: Pedro Ramírez
  ID: P4412C | Nombre: Miguel Torres
  ID: P6678D | Nombre: Ana García
  ID: P7821A | Nombre: Carlos Sánchez
  ID: P9345B | Nombre: Laura Martínez


In [42]:
# Consulta 2: Citas con diagnóstico de Hipertensión (I10)
query_hta = """
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX clinic: <http://www.clinica-bienestar.es/ontology#>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>

SELECT ?nombre ?fecha
WHERE {
  ?p foaf:name ?nombre ;
     clinic:tieneCita ?cita .
  ?cita clinic:fechaCita ?fecha ;
        clinic:conDiagnostico "I10" .
}
ORDER BY ?fecha
"""

print('=== Pacientes con Hipertensión (I10) ===')
for row in g_file.query(query_hta):
    print(f'  Paciente: {row.nombre} | Fecha: {row.fecha}')

=== Pacientes con Hipertensión (I10) ===
  Paciente: Carlos Sánchez | Fecha: 2025-09-15T10:00:00+00:00
  Paciente: Pedro Ramírez | Fecha: 2026-02-28T11:00:00+00:00


In [43]:
# Consulta 3: Número de citas por paciente
query_count = """
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX clinic: <http://www.clinica-bienestar.es/ontology#>

SELECT ?nombre (COUNT(?cita) AS ?numCitas)
WHERE {
  ?p a clinic:Paciente ;
     foaf:name ?nombre ;
     clinic:tieneCita ?cita .
}
GROUP BY ?nombre
ORDER BY DESC(?numCitas)
"""

print('=== Número de citas por paciente ===')
for row in g_file.query(query_count):
    print(f'  {row.nombre}: {row.numCitas} cita(s)')

=== Número de citas por paciente ===
  Carlos Sánchez: 2 cita(s)
  Miguel Torres: 2 cita(s)
  Pedro Ramírez: 2 cita(s)
  Laura Martínez: 1 cita(s)
  Ana García: 1 cita(s)


## 8. Exportar el grafo a diferentes formatos

In [44]:
import os
os.makedirs('output', exist_ok=True)

# Exportar a Turtle
g_file.serialize('output/clinica_bienestar_export.ttl', format='turtle')

# Exportar a JSON-LD
g_file.serialize('output/clinica_bienestar_export.jsonld', format='json-ld', indent=2)

# Exportar a N-Triples
g_file.serialize('output/clinica_bienestar_export.nt', format='nt')

print('✅ Grafo exportado en formatos: Turtle, JSON-LD, N-Triples')
print('   Archivos guardados en /output/')

✅ Grafo exportado en formatos: Turtle, JSON-LD, N-Triples
   Archivos guardados en /output/


/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(
